In [1]:
# ============================================================
# CELL 1 — Imports
# ============================================================
import pandas as pd
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score,
                             balanced_accuracy_score,
                             precision_score, recall_score)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


In [5]:
# ============================================================
# CELL 2 — Load data and filter T1 vs T2 ONLY
# ============================================================

MASTER_PATH  = r"C:\Users\hp\StressDetection\master_features_restored.csv"
RESULTS_PATH = r"C:\Users\hp\StressDetection\ML\loso_results_T1vsT2.csv"

# Stream 1 — geometric features
GEO_FEATURES = [
    'EAR_mean', 'EAR_std', 'EAR_min', 'EAR_max', 'EAR_range',
    'blink_mean', 'blink_std', 'blink_min', 'blink_max', 'blink_range',
    'gaze_mean', 'gaze_std', 'gaze_min', 'gaze_max', 'gaze_range',
    'brow_mean', 'brow_std', 'brow_min', 'brow_max', 'brow_range',
    'lip_mean', 'lip_std', 'lip_min', 'lip_max', 'lip_range'
]

# Stream 2 — valence features
VAL_FEATURES = [
    'valence_mean', 'valence_std', 'valence_min', 'valence_max'
]

# Load full CSV
df_full = pd.read_csv(MASTER_PATH)

# ── FILTER: keep T1 and T2 only ──
df = df_full[df_full['task'].isin(['T1', 'T2'])].copy()

print("=" * 55)
print("T1 vs T2 — DATA SUMMARY")
print("=" * 55)
print(f"Full dataset rows  : {len(df_full)}")
print(f"T1 vs T2 rows      : {len(df)}  (T3 excluded)")
print(f"Participants       : {df['participant'].nunique()}")
print(f"Tasks              : {sorted(df['task'].unique())}")
print(f"Label distribution : {df['label'].value_counts().to_dict()}")
print(f"  label 0 = T1 (relaxed)")
print(f"  label 1 = T2 (arithmetic stress)")
print(f"Stream 1 features  : {len(GEO_FEATURES)}")
print(f"Stream 2 features  : {len(VAL_FEATURES)}")
print("=" * 55)

T1 vs T2 — DATA SUMMARY
Full dataset rows  : 29736
T1 vs T2 rows      : 19824  (T3 excluded)
Participants       : 56
Tasks              : ['T1', 'T2']
Label distribution : {0: 9912, 1: 9912}
  label 0 = T1 (relaxed)
  label 1 = T2 (arithmetic stress)
Stream 1 features  : 25
Stream 2 features  : 4


In [6]:
# ============================================================
# CELL 3 — Late fusion LOSO function (unchanged from original)
# ============================================================
def run_loso_late_fusion(df, clf_name, clf_class, clf_params):
    participants = sorted(df['participant'].unique())
    fold_results = []

    for i, test_pid in enumerate(participants):
        print(f"  [{i+1:02d}/{len(participants)}] Test: p{test_pid}", end='\r')

        train_df = df[df['participant'] != test_pid]
        test_df  = df[df['participant'] == test_pid]

        if test_df['label'].nunique() < 2:
            print(f"\n  WARNING: p{test_pid} has one class — skipping.")
            continue

        X_train_geo = train_df[GEO_FEATURES].values.astype(float)
        X_test_geo  = test_df[GEO_FEATURES].values.astype(float)
        X_train_val = train_df[VAL_FEATURES].values.astype(float)
        X_test_val  = test_df[VAL_FEATURES].values.astype(float)
        y_train     = train_df['label'].values
        y_test      = test_df['label'].values

        # Handle NaN
        for X_tr, X_te in [(X_train_geo, X_test_geo),
                            (X_train_val, X_test_val)]:
            col_means = np.nanmean(X_tr, axis=0)
            for j in range(X_tr.shape[1]):
                X_tr[np.isnan(X_tr[:, j]), j] = col_means[j]
                X_te[np.isnan(X_te[:, j]), j] = col_means[j]

        # Normalize — fit on TRAIN only (no leakage)
        scaler_geo = StandardScaler()
        scaler_val = StandardScaler()
        X_train_geo = scaler_geo.fit_transform(X_train_geo)
        X_test_geo  = scaler_geo.transform(X_test_geo)
        X_train_val = scaler_val.fit_transform(X_train_val)
        X_test_val  = scaler_val.transform(X_test_val)

        # Train both streams
        clf1 = clf_class(**clf_params)
        clf2 = clf_class(**clf_params)
        clf1.fit(X_train_geo, y_train)
        clf2.fit(X_train_val, y_train)

        # Fusion weights from training accuracy
        acc1 = accuracy_score(y_train, clf1.predict(X_train_geo))
        acc2 = accuracy_score(y_train, clf2.predict(X_train_val))
        w1   = acc1 / (acc1 + acc2)
        w2   = acc2 / (acc1 + acc2)

        # Probabilities and fusion
        p1      = clf1.predict_proba(X_test_geo)[:, 1]
        p2      = clf2.predict_proba(X_test_val)[:, 1]
        p_final = w1 * p1 + w2 * p2
        preds   = (p_final >= 0.5).astype(int)

        fold_results.append({
            'participant':  test_pid,
            'classifier':   clf_name,
            'accuracy':     accuracy_score(y_test, preds),
            'f1':           f1_score(y_test, preds, zero_division=0),
            'precision':    precision_score(y_test, preds, zero_division=0),
            'recall':       recall_score(y_test, preds, zero_division=0),
            'balanced_acc': balanced_accuracy_score(y_test, preds),
            'w1': w1, 'w2': w2,
            'n_test': len(y_test)
        })

    print()
    return pd.DataFrame(fold_results)

print("Late fusion LOSO function defined!")

Late fusion LOSO function defined!


In [7]:
# ============================================================
# CELL 4 — Define classifiers (SVM + XGBoost only)
# ============================================================
CLASSIFIERS = {
    'SVM_Linear': (
        CalibratedClassifierCV,
        {
            'estimator': LinearSVC(C=1.0, class_weight='balanced',
                                   random_state=42, max_iter=2000),
            'cv': 3
        }
    ),
    'XGBoost': (
        XGBClassifier,
        {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1,
         'eval_metric': 'logloss', 'random_state': 42, 'verbosity': 0}
    )
}

print("Classifiers defined: SVM_Linear, XGBoost")

Classifiers defined: SVM_Linear, XGBoost


In [8]:
# ============================================================
# CELL 5 — Run late fusion LOSO (T1 vs T2 only)
# ============================================================
all_fold_results = []

for clf_name, (clf_class, clf_params) in CLASSIFIERS.items():
    print(f"\nRunning LOSO — {clf_name}...")
    fold_df = run_loso_late_fusion(df, clf_name, clf_class, clf_params)
    all_fold_results.append(fold_df)

df_all_results = pd.concat(all_fold_results, ignore_index=True)
df_all_results.to_csv(RESULTS_PATH, index=False)
print(f"\nResults saved to: {RESULTS_PATH}")


Running LOSO — SVM_Linear...
  [56/56] Test: p56

Running LOSO — XGBoost...
  [56/56] Test: p56

Results saved to: C:\Users\hp\StressDetection\ML\loso_results_T1vsT2.csv


In [9]:
# ============================================================
# CELL 6 — Results summary + comparison vs Paper [2]
# ============================================================
print("=" * 60)
print("FINAL RESULTS — T1 vs T2 (SVM Linear + XGBoost)")
print("=" * 60)

summary = df_all_results.groupby('classifier').agg(
    Accuracy    =('accuracy',     'mean'),
    F1          =('f1',           'mean'),
    Precision   =('precision',    'mean'),
    Recall      =('recall',       'mean'),
    Balanced_Acc=('balanced_acc', 'mean'),
    Std_Acc     =('accuracy',     'std'),
    Avg_w1      =('w1',           'mean'),
    Avg_w2      =('w2',           'mean'),
).round(4)
print(summary.to_string())

print()
print("=" * 60)
print("COMPARISON vs LITERATURE (T1 vs T2)")
print("=" * 60)

our_svm = df_all_results[df_all_results['classifier']=='SVM_Linear']['accuracy'].mean()*100
our_xgb = df_all_results[df_all_results['classifier']=='XGBoost']['accuracy'].mean()*100

print(f"{'Method':<35} {'Accuracy':>10}  {'Notes'}")
print("-" * 70)
print(f"{'[2] Ziaratnia (WACV 2024) CCT-LSTM':<35} {'98.1%':>10}  7-fold CV, deep learning, black box")
print(f"{'[1] Meziati Sabour 2021 (best)':<35} {'85.48%':>10}  rPPG-SVM, same dataset")
print("-" * 70)
print(f"{'Our SVM Linear (T1 vs T2)':<35} {our_svm:>9.2f}%  LOSO 56-fold, interpretable SHAP")
print(f"{'Our XGBoost   (T1 vs T2)':<35} {our_xgb:>9.2f}%  LOSO 56-fold, interpretable SHAP")
print("-" * 70)
print(f"{'Our SVM Linear (T1 vs T2+T3)':<35} {'92.70%':>10}  Previous result for reference")
print(f"{'Our XGBoost   (T1 vs T2+T3)':<35} {'92.53%':>10}  Previous result for reference")
print()
print(f"Gap vs Ziaratnia (SVM): {98.1 - our_svm:+.2f}%")
print(f"Gap vs Ziaratnia (XGB): {98.1 - our_xgb:+.2f}%")
print()
print("NOTE: Ziaratnia uses deep learning (CCT-LSTM) with 7-fold CV.")
print("Our method uses ML with LOSO (stricter validation) and is fully interpretable.")
print("=" * 60)

FINAL RESULTS — T1 vs T2 (SVM Linear + XGBoost)
            Accuracy      F1  Precision  Recall  Balanced_Acc  Std_Acc  Avg_w1  Avg_w2
classifier                                                                            
SVM_Linear    0.9427  0.9397     0.9655  0.9216        0.9427   0.0545  0.5672  0.4328
XGBoost       0.9457  0.9454     0.9550  0.9418        0.9457   0.0527  0.5571  0.4429

COMPARISON vs LITERATURE (T1 vs T2)
Method                                Accuracy  Notes
----------------------------------------------------------------------
[2] Ziaratnia (WACV 2024) CCT-LSTM       98.1%  7-fold CV, deep learning, black box
[1] Meziati Sabour 2021 (best)          85.48%  rPPG-SVM, same dataset
----------------------------------------------------------------------
Our SVM Linear (T1 vs T2)               94.27%  LOSO 56-fold, interpretable SHAP
Our XGBoost   (T1 vs T2)                94.57%  LOSO 56-fold, interpretable SHAP
------------------------------------------------------